# 2 · Entrenamiento y evaluación

> **Tipo de ML:** `{{ cookiecutter.ml_type }}`


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from {{ cookiecutter.project_module_name }}.utils.paths import PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR

{% if cookiecutter.ml_type == "supervisado" %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models, load_models
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models, predict_new, DECISION_THRESHOLD
from {{ cookiecutter.project_module_name }}.visualization.visualize import plot_feature_importance
{% elif cookiecutter.ml_type == "no_supervisado" %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models
{% elif cookiecutter.ml_type == "redes_neuronales" %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models, load_model
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models
{% endif %}


## 2. Cargar datos procesados


In [ ]:
{% if cookiecutter.ml_type in ["supervisado", "redes_neuronales"] %}
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test  = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test  = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance clases (test): {y_test.value_counts(normalize=True).to_dict()}")
{% elif cookiecutter.ml_type == "no_supervisado" %}
import numpy as np
X = pd.read_csv(PROCESSED_DATA_DIR / "X_processed.csv").values
print(f"Shape: {X.shape}")
{% endif %}


## 3. Entrenar modelos


In [ ]:
{% if cookiecutter.ml_type == "supervisado" %}
# tune_knn=True  → busca el mejor k por cross-validation
# cv_evaluate=True → muestra F1_weighted CV de cada modelo
models = train_models(X_train, y_train, tune_knn=True, cv_evaluate=True)
{% elif cookiecutter.ml_type == "no_supervisado" %}
N_CLUSTERS = 3   # ← ajusta tras ver el gráfico de codo
models = train_models(X, n_clusters=N_CLUSTERS)
{% elif cookiecutter.ml_type == "redes_neuronales" %}
input_dim  = X_train.shape[1]
output_dim = y_train.nunique()
models = train_models(
    X_train, y_train,
    input_dim=input_dim,
    output_dim=output_dim,
    epochs=50,
    batch_size=32,
    checkpoint_every=10,
)
{% endif %}


## 4. Evaluar


In [ ]:
{% if cookiecutter.ml_type == "supervisado" %}
# threshold: umbral de probabilidad para clase positiva
# < 0.5 → más recall  |  > 0.5 → más precisión
threshold = DECISION_THRESHOLD   # modificar si el dataset está desbalanceado

df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test, threshold=threshold
)
df_results
{% elif cookiecutter.ml_type == "no_supervisado" %}
evaluate_models(models, X)
{% elif cookiecutter.ml_type == "redes_neuronales" %}
num_classes = y_test.nunique()
evaluate_models(models, X_test, y_test, num_classes=num_classes)
{% endif %}


## 5. Importancia de variables

{% if cookiecutter.ml_type == 'supervisado' %}Disponible para RandomForest (Gini) y LogisticRegression (|coeficientes|).{% endif %}


In [ ]:
{% if cookiecutter.ml_type == "supervisado" %}
feature_names = X_train.columns.tolist()
plot_feature_importance(models, feature_names)

# Ver la figura generada
from IPython.display import Image
Image(FIGURES_DIR / "feature_importance.png")
{% elif cookiecutter.ml_type == "no_supervisado" or cookiecutter.ml_type == "redes_neuronales" %}
# No disponible para este tipo de ML
pass
{% endif %}


## 6. Comparación de modelos


In [ ]:
{% if cookiecutter.ml_type == "supervisado" %}
# Tabla resumen ya guardada en figures/model_comparison.png
from IPython.display import Image
Image(FIGURES_DIR / "model_comparison.png")
{% elif cookiecutter.ml_type == "supervisado" %}
pass  # Ver figures/clusters_*.png
{% endif %}


## 7. Predicción sobre nuevos datos


In [ ]:
{% if cookiecutter.ml_type == "supervisado" %}
# Ejemplo: predecir con el mejor modelo
best_model_name = df_results.sort_values('Acc_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_model_name]
print(f'Mejor modelo: {best_model_name}')

# Sustituye X_new por tus datos preprocesados
# from {{ cookiecutter.project_module_name }}.features.build_features import process_input
# X_new = process_input(df_nuevos)
# preds, probs = predict_new(best_model, X_new, threshold=threshold)
# print(preds)
{% elif cookiecutter.ml_type == "no_supervisado" %}
# model = joblib.load(MODELS_DIR / 'KMeans.joblib')
# labels = model.predict(X_new)
{% elif cookiecutter.ml_type == "redes_neuronales" %}
# model = load_model(input_dim=input_dim, output_dim=output_dim)
# preds = model(X_tensor)
{% endif %}
